# 04 - Sparsity prior + (F)ISTA

Lecture section: 3.10-3.16  |  Spine term this tutorial changes: the prior $R(x)$

$$\hat{x} = \arg\min_x\ \underbrace{D(Ax, y)}_{\text{data fidelity (fixed)}} + \underbrace{R(x)}_{\text{prior (we change this)}}$$

Same physics $D$ as notebook 3 (sparse-view, noisy CT). Last time the prior was
**Tikhonov**, $R(x)=\tfrac{\lambda}{2}\|x\|^2$, which penalises large pixel values
and only ever **blurs**. Here we swap in a **sparsity prior**:
$R(x)=\lambda\,\mathrm{TV}(x)=\lambda\,\|\nabla x\|_1$, which says *the image has few
edges* (its gradient is sparse). The proximal operator of TV is non-smooth, so we
minimise with **proximal gradient descent (ISTA)** and its accelerated cousin
**FISTA**. Choice: $D=\tfrac12\|Ax-y\|^2$ (L2), $R=\lambda\,\mathrm{TV}$, algorithm = (F)ISTA.

In [1]:
import tutorial_common as tc
import deepinv as dinv
import torch
import math

tc.set_seed()

deepinv 0.4.1 | torch 2.9.1 | device cpu


## The shared problem (identical to notebook 3)

40 projection angles, Gaussian noise $\sigma=0.02$. The operator is normalised so
$\|A^\top A\|\approx 1$, hence the gradient stepsize $1/\|A^\top A\|\approx 1$.

In [2]:
phys = tc.ct_physics(angles=40, sigma=0.02)   # fixed data fidelity D = physics + noise
x = tc.load_hero(128)                          # ground-truth object  (1,1,128,128)
y = phys(x)                                     # noisy sinogram       (1,1,182,40)
stepsize = tc.stepsize_for(phys, x)            # 1 / ||A^T A||  ~ 1.0

print(f"y shape {tuple(y.shape)} | stepsize {stepsize:.3f}")

y shape (1, 1, 182, 40) | stepsize 1.000


## R = Tikhonov vs R = TV  (same D, same algorithm)

We solve the spine equation twice, changing **only the prior** $R$. Both use the
same proximal-gradient solver (`iteration="PGD"`) and the same data fidelity
`L2()` $=\tfrac12\|Ax-y\|^2$. Each prior gets its own tuned $\lambda$ (the knob that
trades data fidelity against the prior).

In [3]:
def reconstruct(prior, lam, max_iter):
    """Solve argmin_x 0.5||Ax-y||^2 + lam * R(x) by proximal gradient (PGD)."""
    model = dinv.optim.optim_builder(
        iteration="PGD",
        data_fidelity=dinv.optim.L2(),                 # D = 0.5||Ax - y||^2  (FIXED)
        prior=prior,                                   # R = the term we change
        params_algo={"lambda": lam, "stepsize": stepsize},
        max_iter=max_iter,
        verbose=False,
    )
    return model(y, phys)

# Tikhonov: smooth L2 prior -> blurs.  TV: sparse-gradient prior -> sharp edges.
x_tik = reconstruct(dinv.optim.Tikhonov(),        lam=0.005, max_iter=100)
x_tv  = reconstruct(dinv.optim.TVPrior(n_it_max=20), lam=0.001, max_iter=300)

print(f"Tikhonov PSNR {tc.psnr(x_tik, x):.2f} dB | TV PSNR {tc.psnr(x_tv, x):.2f} dB")

Tikhonov PSNR 19.43 dB | TV PSNR 21.37 dB


In [4]:
tc.save_images(
    [x, x_tik, x_tv],
    titles=["x (ground truth)",
            tc.title_psnr(r"Tikhonov $R=\lambda||x||^2$", x_tik, x),
            tc.title_psnr(r"TV $R=\lambda||\nabla x||_1$", x_tv, x)],
    fname="04_tv_vs_tikhonov.png",
    suptitle=r"Same data fidelity $D$, different prior $R$",
    figsize=(12, 4),   # wide enough that per-panel titles don't collide
)

saved /Users/jonathan/Code/deepinv/lecture-tutorials/figures/04_tv_vs_tikhonov.png


**Why TV is sharper.** Tikhonov's prox shrinks *every* pixel a little, so it smears
edges into gentle ramps. TV's prox is a soft-threshold on the **image gradient**:
it drives small gradients to exactly zero while leaving large jumps intact. The
result is **piecewise-constant** — flat regions separated by crisp edges, exactly
the structure of our phantom. A sparse-gradient prior recovers edges the L2 prior
can only blur.

## The convergence story: ISTA vs FISTA

TV's prox is non-smooth, so we cannot just run plain gradient descent. **Proximal
gradient (ISTA)** alternates a gradient step on the smooth data term with the
prox of the prior:

$$x_{k+1} = \mathrm{prox}_{\gamma\lambda\, \mathrm{TV}}\!\big(x_k - \gamma\,\nabla D(x_k)\big),
\qquad \gamma=\text{stepsize}.$$

**FISTA** adds one cheap ingredient — a Nesterov **momentum** extrapolation $z_k$ —
which upgrades the convergence rate from $O(1/k)$ to $O(1/k^2)$. We implement both
by hand from the same building blocks: `df.grad(...)` for $\nabla D$ and
`prior.prox(u, gamma=...)` for the TV prox. Note the kwarg is `gamma=`.

In [5]:
df = dinv.optim.L2()                       # D: provides .grad and the cost  D(x,y,phys)
prior = dinv.optim.TVPrior(n_it_max=20)    # R = TV: provides .prox and the value TV(x)
lambd, n_iter = 0.001, 60                   # same lambda as the TV panel above

def cost(x_k):
    """Objective value  D(Ax,y) + lambda * TV(x)  at the current iterate."""
    return (df(x_k, y, phys) + lambd * prior(x_k)).item()

In [6]:
# --- ISTA: proximal gradient, no momentum (O(1/k)) ---
x_k = torch.zeros_like(x)
ista_cost = []
with torch.no_grad():
    for _ in range(n_iter):
        u = x_k - stepsize * df.grad(x_k, y, phys)     # gradient step on D
        x_k = prior.prox(u, gamma=lambd * stepsize)    # prox step on R (TV)
        ista_cost.append(cost(x_k))
ista_psnr = tc.psnr(x_k, x)

# --- FISTA: same two steps + Nesterov momentum z_k (O(1/k^2)) ---
x_k = torch.zeros_like(x)
z_k = x_k.clone()
t_k = 1.0
fista_cost = []
with torch.no_grad():
    for _ in range(n_iter):
        u = z_k - stepsize * df.grad(z_k, y, phys)     # gradient step at the EXTRAPOLATED point
        x_next = prior.prox(u, gamma=lambd * stepsize) # prox step on R (TV)
        t_next = (1.0 + math.sqrt(1.0 + 4.0 * t_k ** 2)) / 2.0
        z_k = x_next + ((t_k - 1.0) / t_next) * (x_next - x_k)   # momentum extrapolation
        x_k, t_k = x_next, t_next
        fista_cost.append(cost(x_k))
fista_psnr = tc.psnr(x_k, x)

print(f"after {n_iter} iters:  ISTA PSNR {ista_psnr:.2f} dB  |  FISTA PSNR {fista_psnr:.2f} dB")
print(f"final cost:           ISTA {ista_cost[-1]:.3f}      |  FISTA {fista_cost[-1]:.3f}")

after 60 iters:  ISTA PSNR 17.90 dB  |  FISTA PSNR 19.39 dB
final cost:           ISTA 3.422      |  FISTA 2.500


In [7]:
tc.save_curves(
    {r"ISTA  ($O(1/k)$)": ista_cost, r"FISTA  ($O(1/k^2)$)": fista_cost},
    fname="04_ista_vs_fista.png",
    xlabel="iteration",
    ylabel="objective cost",
    title="Same problem, momentum converges faster",
    logy=True,
)

saved /Users/jonathan/Code/deepinv/lecture-tutorials/figures/04_ista_vs_fista.png


FISTA's curve sits **below** ISTA's at every single iteration: the momentum term
reuses the previous step's direction to "overshoot" in a controlled way, so it
reaches a low-cost (sharp, accurate) solution in far fewer iterations. Same prox,
same gradient, one extra line of code.

## Takeaway

A **sparsity prior** ($R=\lambda\,\mathrm{TV}$) recovers crisp edges where the L2
(Tikhonov) prior could only blur. And **FISTA's momentum** reaches a good solution
in a fraction of the iterations ISTA needs — for free.